In [ ]:
!pip install requests pandas

##### raw csv

In [1]:
import requests
import pandas as pd
import time

# ==========================================
# 1. YOUR OPENAQ API KEY
# ==========================================

API_KEY = "8d806b0f2a12271bbb12484ede0185bdbad5f8a82995779015c228fa3a98bab6"

headers = {
    "X-API-Key": API_KEY
}

# ==========================================
# 2. SENSOR IDs
# ==========================================

sensors = {
    "PM2.5": 12236457,
    "PM10": 12236456,
    "NO2": 12236454,
    "O3": 12236455,
    "Temperature": 12343701,
    "Humidity": 12343700,
    "Wind Speed": 14342079,
    "Wind Direction": 14342078
}

# ==========================================
# 3. DATE RANGE
# ==========================================

date_from = "2025-02-18"
date_to = "2026-09-02"

# ==========================================
# 4. CREATE MONTHLY DATE CHUNKS
# ==========================================

dates = pd.date_range(
    start=date_from,
    end=date_to,
    freq="MS"
)

all_data = []

# ==========================================
# 5. DOWNLOAD DATA MONTH BY MONTH
# ==========================================

for variable, sensor_id in sensors.items():

    print("\n====================================")
    print(f"Downloading: {variable}")
    print(f"Sensor ID: {sensor_id}")
    print("====================================")

    for i in range(len(dates)):

        chunk_start = dates[i]

        if i + 1 < len(dates):
            chunk_end = dates[i + 1] - pd.Timedelta(seconds=1)
        else:
            chunk_end = pd.Timestamp(date_to) + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)

        start_str = chunk_start.strftime("%Y-%m-%dT%H:%M:%S")
        end_str = chunk_end.strftime("%Y-%m-%dT%H:%M:%S")

        print(
            f"{chunk_start.strftime('%Y-%m-%d')} → "
            f"{chunk_end.strftime('%Y-%m-%d')}"
        )

        page = 1

        while True:

            url = f"https://api.openaq.org/v3/sensors/{sensor_id}/hours"

            params = {
                "datetime_from": start_str,
                "datetime_to": end_str,
                "limit": 1000,
                "page": page
            }

            try:

                response = requests.get(
                    url,
                    headers=headers,
                    params=params,
                    timeout=60
                )

                print(
                    f"   Page {page} | "
                    f"Status: {response.status_code}"
                )

                response.raise_for_status()

                data = response.json()

                results = data.get("results", [])

                if not results:
                    break

                for row in results:
                    row["variable"] = variable
                    row["sensor_id"] = sensor_id
                    all_data.append(row)

                if len(results) < 1000:
                    break

                page += 1

                time.sleep(0.5)

            except requests.exceptions.HTTPError as e:

                print(
                    f"   ERROR on page {page}: {e}"
                )

                print("   Waiting 10 seconds and retrying...")

                time.sleep(10)

                continue

# ==========================================
# 6. CREATE DATAFRAME
# ==========================================

hourly_df = pd.json_normalize(all_data)

print("\n====================================")
print("HOURLY DOWNLOAD COMPLETE")
print("====================================")

print("Total records:", len(hourly_df))

print("\nColumns:")
print(hourly_df.columns.tolist())

# ==========================================
# 7. SAVE CSV
# ==========================================

hourly_df.to_csv(
    "Ravet_Nigdi_OpenAQ_Hourly_Data.csv",
    index=False
)

print("\nCSV created successfully:")
print("Ravet_Nigdi_OpenAQ_Hourly_Data.csv")


Downloading: PM2.5
Sensor ID: 12236457
2025-03-01 → 2025-03-31
   Page 1 | Status: 200
2025-04-01 → 2025-04-30
   Page 1 | Status: 200
2025-05-01 → 2025-05-31
   Page 1 | Status: 200
2025-06-01 → 2025-06-30
   Page 1 | Status: 200
2025-07-01 → 2025-07-31
   Page 1 | Status: 200
2025-08-01 → 2025-08-31
   Page 1 | Status: 200
2025-09-01 → 2025-09-30
   Page 1 | Status: 200
2025-10-01 → 2025-10-31
   Page 1 | Status: 200
2025-11-01 → 2025-11-30
   Page 1 | Status: 200
2025-12-01 → 2025-12-31
   Page 1 | Status: 200
2026-01-01 → 2026-01-31
   Page 1 | Status: 200
2026-02-01 → 2026-02-28
   Page 1 | Status: 200
2026-03-01 → 2026-03-31
   Page 1 | Status: 200
2026-04-01 → 2026-04-30
   Page 1 | Status: 200
2026-05-01 → 2026-05-31
   Page 1 | Status: 200
2026-06-01 → 2026-06-30
   Page 1 | Status: 200
2026-07-01 → 2026-07-31
   Page 1 | Status: 200
2026-08-01 → 2026-08-31
   Page 1 | Status: 200
2026-09-01 → 2026-09-02
   Page 1 | Status: 200

Downloading: PM10
Sensor ID: 12236456
2025-03-0

##### cleaned csv

In [3]:
import pandas as pd

# df is already available from your download cell.
# If Jupyter was restarted, uncomment this line:
df = pd.read_csv("Ravet_Nigdi_OpenAQ_Hourly_Data.csv")

required_parameters = [
    "pm25", "pm10", "no2", "o3",
    "temperature", "relativehumidity",
    "wind_speed", "wind_direction"
]

# Keep required variables
cleaned_df = df[df["parameter.name"].isin(required_parameters)].copy()

# Convert timestamp and use the hourly average
cleaned_df["timestamp"] = pd.to_datetime(
    cleaned_df["period.datetimeFrom.local"],
    errors="coerce"
)

cleaned_df["hourly_value"] = pd.to_numeric(
    cleaned_df["summary.avg"],
    errors="coerce"
)

# Use value if summary.avg is missing
cleaned_df["hourly_value"] = cleaned_df["hourly_value"].fillna(
    pd.to_numeric(cleaned_df["value"], errors="coerce")
)

# Remove rows without usable time/value
cleaned_df = cleaned_df.dropna(
    subset=["timestamp", "hourly_value"]
).copy()

# Remove impossible negative pollutant values
pollutants = ["pm25", "pm10", "no2", "o3"]
cleaned_df.loc[
    cleaned_df["parameter.name"].isin(pollutants) &
    (cleaned_df["hourly_value"] < 0),
    "hourly_value"
] = pd.NA

cleaned_df = cleaned_df.dropna(subset=["hourly_value"])
cleaned_df = cleaned_df.sort_values(["timestamp", "parameter.name"])

cleaned_df.to_csv(
    "Nigdi_OpenAQ_Hourly_Cleaned.csv",
    index=False
)

print("Created: Nigdi_OpenAQ_Hourly_Cleaned.csv")
print("Rows:", len(cleaned_df))

Created: Nigdi_OpenAQ_Hourly_Cleaned.csv
Rows: 75047


##### model ready csv

In [ ]:
# Keep the three fields needed for ML preparation
model_df = cleaned_df[
    ["timestamp", "parameter.name", "hourly_value"]
].copy()

# Convert long format to wide format:
# one row per hour, one column per variable
model_df = model_df.pivot_table(
    index="timestamp",
    columns="parameter.name",
    values="hourly_value",
    aggfunc="mean"
).reset_index()

model_df.columns.name = None

# Rename relativehumidity to humidity
model_df = model_df.rename(columns={
    "relativehumidity": "humidity"
})

# Sort by time
model_df = model_df.sort_values("timestamp").reset_index(drop=True)

# Save the final dataset for ML training
model_df.to_csv(
    "Nigdi_OpenAQ_Hourly_Model_Ready.csv",
    index=False
)

print("Created: Nigdi_OpenAQ_Hourly_Model_Ready.csv")
print("Shape:", model_df.shape)

print("\nMissing values:")
print(model_df.isnull().sum())

print(model_df.head())

Created: Nigdi_OpenAQ_Hourly_Model_Ready.csv
Shape: (10976, 9)

Missing values:
timestamp            0
no2                 28
o3                  32
pm10               185
pm25                33
humidity          5597
temperature       5683
wind_direction     602
wind_speed         601
dtype: int64


,timestamp,no2,o3,pm10,pm25,humidity,temperature,wind_direction,wind_speed
0,2025-03-01 00:00:00+05:30,21.870000,21.37,161.460000,56.4600,NaN,NaN,234.390000,0.070
1,2025-03-01 01:00:00+05:30,30.526667,0.70,171.993333,66.4400,NaN,NaN,309.006667,0.060
2,2025-03-01 02:00:00+05:30,34.322500,0.00,173.610000,69.9275,NaN,NaN,298.817500,0.065
3,2025-03-01 03:00:00+05:30,36.420000,0.00,174.730000,61.9950,NaN,NaN,256.175000,0.065
4,2025-03-01 04:00:00+05:30,42.607500,0.00,180.825000,66.8025,NaN,NaN,204.692500,0.060
